# ASG Airlines - Task 1: Data IngestionLocal Bronze layer. Reads the source workbook, runs data quality checks while loading,keeps every row, and separates clean rows from rejected rows.Same design ports to Databricks later: pandas to PySpark, Parquet to Delta, local folder to Unity Catalog volume.

## Cell 1 - SetupLoad libraries and set the source workbook and the output folders.

In [1]:
import re
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

BASE_DIR = Path.cwd().parent
SOURCE_FILE = BASE_DIR / "Airlines Use Case" / "UseCase - Airlines.xlsx"
BRONZE_DIR = BASE_DIR / "data" / "bronze"
QUARANTINE_DIR = BASE_DIR / "data" / "quarantine"

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_TS = datetime.now(timezone.utc)

## Cell 2 - LoggingA plain console logger so each ingestion step prints what it did.

In [2]:
logger = logging.getLogger("ingestion")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s  %(levelname)s  %(message)s"))
    logger.addHandler(handler)

logger.info("Batch %s", BATCH_ID)
logger.info("Source %s", SOURCE_FILE.name)

2026-09-06 15:00:40,422  INFO  Batch 20260906T093040Z


2026-09-06 15:00:40,423  INFO  Source UseCase - Airlines.xlsx


## Cell 3 - Read sourceRead all four sheets as text so no value is auto-converted, rounded or reformatted.

In [3]:
sheets = pd.read_excel(SOURCE_FILE, sheet_name=None, dtype=object)

raw = {}
for name, frame in sheets.items():
    key = str(name).strip().lower()
    frame = frame.rename(columns={c: str(c).strip().lower() for c in frame.columns})
    raw[key] = frame
    logger.info("Sheet %-12s %5d rows  %d cols", key, len(frame), frame.shape[1])

list(raw)

2026-09-06 15:00:41,549  INFO  Sheet flights       1020 rows  7 cols


2026-09-06 15:00:41,551  INFO  Sheet payments      1000 rows  4 cols


2026-09-06 15:00:41,553  INFO  Sheet bookings      1000 rows  9 cols


2026-09-06 15:00:41,555  INFO  Sheet passengers    1039 rows  9 cols


['flights', 'payments', 'bookings', 'passengers']

## Cell 4 - RulesThe contract for each dataset: key column, required fields, and format checks.

In [4]:
RULES = {
    "flights": {
        "columns": ["flight_id", "airline", "source", "destination", "departure_time", "arrival_time", "duration"],
        "primary_key": "flight_id",
        "required": ["flight_id", "airline", "source", "destination", "departure_time", "arrival_time"],
        "formats": {
            "flight_id": r"^[0-9A-Z]{2}\d{3,4}$",
            "source": r"^[A-Z]{3}$",
            "destination": r"^[A-Z]{3}$",
        },
        "numeric": [],
    },
    "passengers": {
        "columns": ["passenger_id", "first_name", "last_name", "age", "gender", "email", "phone", "aadhaar_id", "date_of_birth"],
        "primary_key": "passenger_id",
        "required": ["passenger_id", "first_name", "last_name"],
        "formats": {
            "passenger_id": r"^P\d+$",
            "email": r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
            "phone": r"^\+91-\d{10}$",
            "aadhaar_id": r"^\d{12}$",
        },
        "numeric": ["age"],
    },
    "bookings": {
        "columns": ["booking_id", "passenger_id", "flight_id", "booking_date", "status", "passport_number", "seat_number", "emergency_contact_name", "emergency_contact_phone"],
        "primary_key": "booking_id",
        "required": ["booking_id", "passenger_id", "flight_id"],
        "formats": {
            "booking_id": r"^B\d+$",
            "passenger_id": r"^P\d+$",
            "status": r"^(CONFIRMED|CANCELLED|PENDING)$",
            "seat_number": r"^\d{1,2}[A-F]$",
        },
        "numeric": [],
    },
    "payments": {
        "columns": ["payment_id", "booking_id", "amount", "payment_method"],
        "primary_key": "payment_id",
        "required": ["payment_id", "booking_id", "amount"],
        "formats": {
            "payment_id": r"^PAY\d+$",
            "booking_id": r"^B\d+$",
            "payment_method": r"^(UPI|NETBANKING|CARD)$",
        },
        "numeric": ["amount"],
    },
}

## Cell 5 - HelpersTrim each value to a clean string or None, list the tokens that really mean "missing",and hash a row so exact duplicates can be found.

In [5]:
PLACEHOLDERS = {"UNKNOWN", "N/A", "NA", "NULL", "NONE", "-", "?"}

def clean(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    text = str(value).strip()
    return text or None

def row_hash(values):
    payload = json.dumps(values, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()

## Cell 6 - Quality checkWalk every row of one dataset, flag missing values, placeholder values, bad formats,non numeric amounts and repeated keys. Quarantine only when the key is missing or the whole row repeats.

In [6]:
def evaluate(name, frame):
    spec = RULES[name]

    for column in spec["columns"]:
        if column not in frame.columns:
            frame[column] = None
    frame = frame[spec["columns"]]

    records = []
    for position, row in enumerate(frame.itertuples(index=False), start=2):
        values = {col: clean(val) for col, val in zip(spec["columns"], row)}
        issues = []

        for column in spec["required"]:
            if values[column] is None:
                issues.append("missing:" + column)

        for column in spec["columns"]:
            current = values[column]
            if current is not None and current.upper() in PLACEHOLDERS:
                issues.append("placeholder:" + column)

        for column, pattern in spec["formats"].items():
            current = values[column]
            if current is not None and current.upper() not in PLACEHOLDERS and re.match(pattern, current) is None:
                issues.append("bad_format:" + column)

        for column in spec["numeric"]:
            current = values[column]
            if current is not None:
                try:
                    float(current)
                except ValueError:
                    issues.append("not_numeric:" + column)

        record = dict(values)
        record["source_row"] = position
        record["row_hash"] = row_hash(list(values.values()))
        record["dq_issues"] = ";".join(issues)
        records.append(record)

    result = pd.DataFrame(records)
    key = spec["primary_key"]

    key_missing = result[key].isna()
    duplicate_key = result.duplicated(subset=[key], keep="first") & ~key_missing
    duplicate_row = result.duplicated(subset=["row_hash"], keep="first")

    issue_list = []
    reason_list = []
    for i in result.index:
        issues = [x for x in result.at[i, "dq_issues"].split(";") if x]
        if duplicate_key.at[i]:
            issues.append("duplicate_key")

        reasons = []
        if key_missing.at[i]:
            reasons.append("key_missing")
        if duplicate_row.at[i]:
            reasons.append("duplicate_row")

        issue_list.append(";".join(issues))
        reason_list.append(";".join(reasons))

    result["dq_issues"] = issue_list
    result["quarantine_reason"] = reason_list
    result["is_valid"] = result["quarantine_reason"] == ""
    return result

## Cell 7 - Run the checksApply the evaluation to flights, passengers, bookings and payments.

In [7]:
evaluated = {}
for name in RULES:
    checked = evaluate(name, raw[name])
    evaluated[name] = checked
    logger.info(
        "%-12s valid %4d  quarantine %3d  flagged %3d",
        name,
        int(checked["is_valid"].sum()),
        int((~checked["is_valid"]).sum()),
        int((checked["dq_issues"] != "").sum()),
    )

2026-09-06 15:00:41,735  INFO  flights      valid 1005  quarantine  15  flagged  84


2026-09-06 15:00:41,835  INFO  passengers   valid 1039  quarantine   0  flagged  39


2026-09-06 15:00:41,915  INFO  bookings     valid 1000  quarantine   0  flagged  30


2026-09-06 15:00:41,999  INFO  payments     valid 1000  quarantine   0  flagged  78


## Cell 8 - LineageStamp every row with the batch id, source file, sheet name and load time.

In [8]:
def stamp(name, frame):
    frame = frame.copy()
    frame["batch_id"] = BATCH_ID
    frame["source_file"] = SOURCE_FILE.name
    frame["source_sheet"] = name
    frame["ingested_at"] = RUN_TS.isoformat()
    return frame

evaluated = {name: stamp(name, frame) for name, frame in evaluated.items()}

## Cell 9 - Write outputsSave valid rows to the bronze folder and rejected rows to quarantine, as Parquet.

In [9]:
load_summary = []
for name, frame in evaluated.items():
    valid_rows = frame[frame["is_valid"]].drop(columns=["is_valid"])
    quarantine_rows = frame[~frame["is_valid"]].drop(columns=["is_valid"])

    valid_rows.to_parquet(BRONZE_DIR / (name + ".parquet"), index=False)
    quarantine_rows.to_parquet(QUARANTINE_DIR / (name + ".parquet"), index=False)

    load_summary.append({
        "dataset": name,
        "source": len(frame),
        "bronze": len(valid_rows),
        "quarantine": len(quarantine_rows),
        "flagged": int((valid_rows["dq_issues"] != "").sum()),
    })

pd.DataFrame(load_summary)

,dataset,source,bronze,quarantine,flagged
0,flights,1020,1005,15,69
1,passengers,1039,1039,0,39
2,bookings,1000,1000,0,30
3,payments,1000,1000,0,78


## Cell 10 - Referential integrityCount child keys with no matching parent across the ingested tables.

In [10]:
links = [
    ("bookings", "passenger_id", "passengers", "passenger_id"),
    ("bookings", "flight_id", "flights", "flight_id"),
    ("payments", "booking_id", "bookings", "booking_id"),
]

integrity = []
for child, child_key, parent, parent_key in links:
    child_values = evaluated[child][child_key].dropna()
    parent_values = set(evaluated[parent][parent_key].dropna())
    orphans = int((~child_values.isin(parent_values)).sum())
    integrity.append({
        "relation": child + "." + child_key + " -> " + parent,
        "checked": len(child_values),
        "orphans": orphans,
    })

pd.DataFrame(integrity)

,relation,checked,orphans
0,bookings.passenger_id -> passengers,1000,0
1,bookings.flight_id -> flights,1000,0
2,payments.booking_id -> bookings,1000,0


## Cell 11 - Quality reportOne table of every issue type and the row count it touched, saved for the documentation.

In [11]:
report = []
for name, frame in evaluated.items():
    issues = frame["dq_issues"].str.split(";").explode()
    issues = issues[issues.notna() & (issues != "")]
    for issue, count in issues.value_counts().items():
        report.append({"dataset": name, "check": issue, "rows": int(count)})
    for reason, count in frame["quarantine_reason"].value_counts().items():
        if reason:
            report.append({"dataset": name, "check": "quarantine:" + reason, "rows": int(count)})

report_df = pd.DataFrame(report).sort_values(["dataset", "check"]).reset_index(drop=True)
report_df.to_csv(BASE_DIR / "data" / "dq_report.csv", index=False)
report_df

,dataset,check,rows
0,bookings,bad_format:status,30
1,flights,duplicate_key,16
2,flights,missing:airline,41
3,flights,placeholder:airline,31
4,flights,quarantine:duplicate_row,15
5,passengers,duplicate_key,39
6,passengers,missing:last_name,10
7,payments,missing:amount,48
8,payments,not_numeric:amount,30


## Cell 12 - PreviewA quick look at ingested flights and at the flight rows that were quarantined.

In [12]:
from IPython.display import display

display(pd.read_parquet(BRONZE_DIR / "flights.parquet").head(10))
display(pd.read_parquet(QUARANTINE_DIR / "flights.parquet").head(10))

,flight_id,airline,source,destination,departure_time,arrival_time,duration,source_row,row_hash,dq_issues,quarantine_reason,batch_id,source_file,source_sheet,ingested_at
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701000,2026-04-21 02:32:41.701000,02:54:00,2,4a622e7cb8b9070d580ab140d42604cddb36236df4ce83...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703000,2026-04-21 01:23:41.703000,01:48:00,3,479dcae3a14c7168b23201ea322459eab4d245321e5337...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702000,2026-04-21 01:11:41.702000,01:45:00,4,8571f84e19efee78b9318e44935875d409dc14f9110237...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704000,2026-04-21 01:43:41.704000,02:36:00,5,38571723a1a1d898419591fe2ced4e7b00a056a934808f...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703000,2026-04-21 04:04:41.703000,04:59:00,6,381299187783f2c19c2d36e9ecc51693add7b5212524d5...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
5,SJ158,SpiceJet,DEL,HYD,2026-04-20 23:05:41.703000,2026-04-21 01:24:41.703000,02:19:00,7,ca99dd11c74bb0f845ff64dbecbbe67ed2bd5dd8ac5bf3...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
6,6F196,IndiGo,CCU,MAA,2026-04-20 23:04:41.703000,2026-04-21 00:47:41.703000,01:43:00,8,69c8e785a8eb6f4cf4a057e1726d91a079efea1a1d9674...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
7,AI080,Air India,BOM,HYD,2026-04-20 23:03:41.702000,2026-04-21 00:35:41.702000,01:32:00,9,4615dde891b865ddd7488261df9669b9f0efcb43e804a9...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
8,6F025,IndiGo,BLR,BOM,2026-04-20 23:02:41.701000,2026-04-20 23:57:41.701000,00:55:00,10,377512b866346b96d17f399c0f63f5c4679910291e5ace...,,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
9,6F251,UNKNOWN,DEL,BLR,2026-04-20 22:56:41.703000,2026-04-20 23:37:41.703000,00:41:00,11,f3728cede9efa00d36d7ff52d83a9501c615228f11e43c...,placeholder:airline,,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00


,flight_id,airline,source,destination,departure_time,arrival_time,duration,source_row,row_hash,dq_issues,quarantine_reason,batch_id,source_file,source_sheet,ingested_at
0,AI242,UNKNOWN,BLR,CCU,2026-04-20 16:41:41.704000,2026-04-20 17:43:41.704000,01:02:00,86,2a1607436d3a504abdeba110c750d474eff5b13c90a463...,placeholder:airline;duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
1,SJ142,SpiceJet,CCU,BOM,2026-04-20 15:14:41.702000,2026-04-20 16:56:41.702000,01:42:00,108,1daf72e09286c38580deb26ab10f135b606ec985fb9025...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
2,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701000,2026-04-20 16:11:41.701000,03:06:00,144,d95d21abc6ac277768d660d33fec2af2f28cf55a667ed3...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
3,SJ146,SpiceJet,HYD,BLR,2026-04-19 22:37:41.702000,2026-04-20 00:33:41.702000,01:56:00,312,b73b0b5f56cd5b997d78e581f6b74fd8c9eabaad08b63a...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
4,UK163,Vistara,DEL,HYD,2026-04-19 07:22:41.703000,2026-04-19 09:04:41.703000,01:42:00,505,4ab3090442a1fe4e25d48a46a4e881d464c5b00241d2de...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
5,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701000,2026-04-19 06:21:41.701000,02:28:00,552,979ce5f659b2fa019d4815f5ddb0aee6e62228cd4c8fa7...,missing:airline;duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
6,SJ118,SpiceJet,DEL,BOM,2026-04-19 01:08:41.702000,2026-04-19 02:33:41.702000,01:25:00,591,b1a14fdd572b63729697c4770b3f42a4bef9a3af6c9d87...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
7,AI043,Air India,CCU,DEL,2026-04-19 00:28:41.701000,2026-04-19 04:37:41.701000,04:09:00,601,5537e7407f26a9f1224d6566b384268fc5676110b3952e...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
8,AI070,Air India,CCU,DEL,2026-04-19 00:05:41.702000,2026-04-19 01:44:41.702000,01:39:00,605,4425a08411bfa928b2e152f371f9624624e637059ab761...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00
9,UK013,Vistara,BOM,MAA,2026-04-18 23:52:41.701000,2026-04-19 02:52:41.701000,03:00:00,608,3f36aec4dd1aea82dfcfd929f91ad07f88375a0d480280...,duplicate_key,duplicate_row,20260906T093040Z,UseCase - Airlines.xlsx,flights,2026-09-06T09:30:40.409965+00:00


## Cell 13 - Handoff to Task 2Rows are kept and flagged for the transformation stage. The next stage should:- convert the departure time, arrival time and duration into real timestamps- roll overnight arrivals to the next day and recompute the duration- fill the airline where it shows UNKNOWN or is missing- collapse the repeated flight and passenger rows into one record each- fix the non numeric and missing amount values in payments- mask passenger personal data such as email, phone, aadhaar id, passport number and names

# Task 2 - Data Transformation and CleaningRead the bronze tables from Task 1 and turn them into a clean silver layer:duplicates removed, missing values filled, values standardised, and passengerpersonal data replaced with salted hashes.

## Cell 14 - SetupPoint at the silver output folder, set the hashing salt, and load the four bronze tables.

In [13]:
SILVER_DIR = BASE_DIR / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

HASH_SALT = "asg-airlines-2026"

bronze = {name: pd.read_parquet(BRONZE_DIR / (name + ".parquet")) for name in RULES}
{name: len(frame) for name, frame in bronze.items()}

{'flights': 1005, 'passengers': 1039, 'bookings': 1000, 'payments': 1000}

## Cell 15 - Working copiesKeep only the business columns plus the source row number, drop the ingestion scaffolding.

In [14]:
clean_data = {}
for name in bronze:
    columns = [c for c in RULES[name]["columns"] if c in bronze[name].columns]
    clean_data[name] = bronze[name][columns + ["source_row"]].copy()

{name: list(frame.columns) for name, frame in clean_data.items()}

{'flights': ['flight_id',
  'airline',
  'source',
  'destination',
  'departure_time',
  'arrival_time',
  'duration',
  'source_row'],
 'passengers': ['passenger_id',
  'first_name',
  'last_name',
  'age',
  'gender',
  'email',
  'phone',
  'aadhaar_id',
  'date_of_birth',
  'source_row'],
 'bookings': ['booking_id',
  'passenger_id',
  'flight_id',
  'booking_date',
  'status',
  'passport_number',
  'seat_number',
  'emergency_contact_name',
  'emergency_contact_phone',
  'source_row'],
 'payments': ['payment_id',
  'booking_id',
  'amount',
  'payment_method',
  'source_row']}

## Cell 16 - Standardise valuesTrim every text field, upper case the airport codes, and upper case the status and payment method.

In [15]:
def tidy_text(frame):
    for column in frame.columns:
        if frame[column].dtype == object:
            frame[column] = frame[column].astype("string").str.strip()
    return frame

for name in clean_data:
    clean_data[name] = tidy_text(clean_data[name])

clean_data["flights"]["source"] = clean_data["flights"]["source"].str.upper()
clean_data["flights"]["destination"] = clean_data["flights"]["destination"].str.upper()
clean_data["bookings"]["status"] = clean_data["bookings"]["status"].str.upper()
clean_data["payments"]["payment_method"] = clean_data["payments"]["payment_method"].str.upper()

## Cell 17 - Fix the airlineEvery flight code starts with a two letter airline prefix, so fill the missing and UNKNOWNairline names from that prefix and keep the prefix as an airline code.

In [16]:
AIRLINE_BY_PREFIX = {"6F": "IndiGo", "AI": "Air India", "SJ": "SpiceJet", "UK": "Vistara"}

flights = clean_data["flights"]
flights["airline_code"] = flights["flight_id"].str[:2]

missing_airline = flights["airline"].isna() | (flights["airline"].str.upper() == "UNKNOWN")
flights.loc[missing_airline, "airline"] = flights.loc[missing_airline, "airline_code"].map(AIRLINE_BY_PREFIX)
airline_filled = int(missing_airline.sum())

clean_data["flights"] = flights
airline_filled

69

## Cell 18 - Timestamps and overnight flightsParse departure and arrival into real timestamps. When the arrival lands before thedeparture the flight crossed midnight, so push the arrival forward by one day.

In [17]:
flights = clean_data["flights"]
flights["departure_time"] = pd.to_datetime(flights["departure_time"], format="mixed", errors="coerce")
flights["arrival_time"] = pd.to_datetime(flights["arrival_time"], format="mixed", errors="coerce")

crossed_midnight = flights["arrival_time"] < flights["departure_time"]
flights.loc[crossed_midnight, "arrival_time"] = flights.loc[crossed_midnight, "arrival_time"] + pd.Timedelta(days=1)

flights["is_overnight"] = flights["arrival_time"].dt.normalize() > flights["departure_time"].dt.normalize()
clean_data["flights"] = flights

int(flights["is_overnight"].sum())

122

## Cell 19 - Recompute durationReplace the reported duration with the real gap between arrival and departure, in minutes.

In [18]:
flights = clean_data["flights"]
flights["duration_minutes"] = (
    (flights["arrival_time"] - flights["departure_time"]).dt.total_seconds().div(60).round().astype("Int64")
)
flights = flights.drop(columns=["duration"])
clean_data["flights"] = flights

flights[["flight_id", "departure_time", "arrival_time", "duration_minutes", "is_overnight"]].head()

,flight_id,departure_time,arrival_time,duration_minutes,is_overnight
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174,True
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108,True
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105,True
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156,True
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299,True


## Cell 20 - Remove duplicate keysFor each repeated flight id or passenger id keep the row with the fewest blank or placeholderfields, breaking ties by the original source order.

In [19]:
def gap_count(frame, columns):
    text = frame[columns].astype("string")
    blank = text.isna() | text.apply(lambda col: col.str.upper().isin(PLACEHOLDERS))
    return blank.sum(axis=1)

def keep_best(frame, key):
    columns = [c for c in frame.columns if c != "source_row"]
    frame = frame.copy()
    frame["gaps"] = gap_count(frame, columns)
    frame = frame.sort_values(["gaps", "source_row"]).drop_duplicates(subset=[key], keep="first")
    return frame.drop(columns=["gaps"]).sort_values("source_row").reset_index(drop=True)

before = {name: len(clean_data[name]) for name in clean_data}
clean_data["flights"] = keep_best(clean_data["flights"], "flight_id")
clean_data["passengers"] = keep_best(clean_data["passengers"], "passenger_id")

{name: before[name] - len(clean_data[name]) for name in clean_data}

{'flights': 1, 'passengers': 39, 'bookings': 0, 'payments': 0}

## Cell 21 - Fill missing valuesGive an unusable booking status the value UNKNOWN, turn the amount into a number andfill the gaps with the median paid, and fill a missing surname with UNKNOWN.

In [20]:
known_status = ["CONFIRMED", "CANCELLED", "PENDING"]
bookings = clean_data["bookings"]
bookings["status"] = bookings["status"].where(bookings["status"].isin(known_status), "UNKNOWN")
status_fixed = int((bookings["status"] == "UNKNOWN").sum())
clean_data["bookings"] = bookings

payments = clean_data["payments"]
payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
amount_missing = int(payments["amount"].isna().sum())
median_amount = round(payments["amount"].median(), 2)
payments["amount"] = payments["amount"].fillna(median_amount).round(2)
clean_data["payments"] = payments

passengers = clean_data["passengers"]
passengers["last_name"] = passengers["last_name"].fillna("UNKNOWN")
clean_data["passengers"] = passengers

{"status_set_unknown": status_fixed, "amount_imputed": amount_missing, "median_amount": median_amount}

{'status_set_unknown': 75,
 'amount_imputed': 78,
 'median_amount': np.float64(8027.12)}

## Cell 22 - Hash personal dataReplace every personal field with a salted SHA-256 hash. The same input always gives thesame hash, so rows still join and group, but no readable name, contact or id remains.The date of birth is dropped because the age column already covers it.

In [21]:
def hash_value(value):
    if pd.isna(value):
        return None
    return hashlib.sha256((HASH_SALT + str(value)).encode()).hexdigest()[:16]

PII_COLUMNS = {
    "passengers": ["first_name", "last_name", "email", "phone", "aadhaar_id"],
    "bookings": ["passport_number", "emergency_contact_name", "emergency_contact_phone"],
}

for name, columns in PII_COLUMNS.items():
    frame = clean_data[name]
    for column in columns:
        if column in frame.columns:
            frame[column] = frame[column].map(hash_value)
    clean_data[name] = frame

if "date_of_birth" in clean_data["passengers"].columns:
    clean_data["passengers"] = clean_data["passengers"].drop(columns=["date_of_birth"])

clean_data["passengers"].head()

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,source_row
0,P1000,c1016bac4d62c349,0898a357873bdcfd,52,F,90a594d3d2618faa,66d2db762284b8f3,dbc976463c58cebc,2
1,P1001,64c7e7a78fe65a03,5c012c348379ffe8,15,M,3289671730c1cbb6,eb29bca6561ac5c3,770d0e3801d16861,3
2,P1002,163bb7b35d57ad33,a1c739a678cb4d19,72,M,1b71dfbe84e5e118,dd17a313cdbb20e0,d94d5c74b1b87dfe,4
3,P1003,163bb7b35d57ad33,25e29e98d33c67f8,61,F,da1347ef9a577681,a7bb9660ea46cc46,c0365e092e3fd791,5
4,P1004,6d1eca2f30004fda,4cb38c56eea0ddf5,21,M,6e9695bce76a8f63,26831a7237109534,e15e595420702d9c,6


## Cell 23 - Referential integrityCheck that every foreign key in the cleaned tables still points at a real parent row.

In [22]:
silver_links = [
    ("bookings", "passenger_id", "passengers", "passenger_id"),
    ("bookings", "flight_id", "flights", "flight_id"),
    ("payments", "booking_id", "bookings", "booking_id"),
]

integrity_after = []
for child, child_key, parent, parent_key in silver_links:
    child_values = clean_data[child][child_key].dropna()
    parent_values = set(clean_data[parent][parent_key].dropna())
    orphans = int((~child_values.isin(parent_values)).sum())
    integrity_after.append({"relation": child + "." + child_key + " -> " + parent, "orphans": orphans})

pd.DataFrame(integrity_after)

,relation,orphans
0,bookings.passenger_id -> passengers,0
1,bookings.flight_id -> flights,0
2,payments.booking_id -> bookings,0


## Cell 24 - Write the silver layerDrop the source row helper and save each cleaned table as Parquet in the silver folder.

In [23]:
for name, frame in clean_data.items():
    frame.drop(columns=["source_row"]).to_parquet(SILVER_DIR / (name + ".parquet"), index=False)
    logger.info("silver %-12s %5d rows  %d cols", name, len(frame), frame.shape[1] - 1)

{name: frame.drop(columns=["source_row"]).shape for name, frame in clean_data.items()}

2026-09-06 15:00:42,899  INFO  silver flights       1004 rows  9 cols


2026-09-06 15:00:42,910  INFO  silver passengers    1000 rows  8 cols


2026-09-06 15:00:42,921  INFO  silver bookings      1000 rows  9 cols


2026-09-06 15:00:42,929  INFO  silver payments      1000 rows  4 cols


{'flights': (1004, 9),
 'passengers': (1000, 8),
 'bookings': (1000, 9),
 'payments': (1000, 4)}

## Cell 25 - Cleaning summaryOne table showing what changed between bronze and silver, saved for the documentation.

In [24]:
change_log = [
    {"dataset": "flights", "change": "rows in bronze", "count": len(bronze["flights"])},
    {"dataset": "flights", "change": "rows in silver", "count": len(clean_data["flights"])},
    {"dataset": "flights", "change": "airline filled from code", "count": airline_filled},
    {"dataset": "flights", "change": "overnight flights", "count": int(clean_data["flights"]["is_overnight"].sum())},
    {"dataset": "passengers", "change": "rows in bronze", "count": len(bronze["passengers"])},
    {"dataset": "passengers", "change": "rows in silver", "count": len(clean_data["passengers"])},
    {"dataset": "bookings", "change": "status set to UNKNOWN", "count": status_fixed},
    {"dataset": "payments", "change": "amount imputed to median", "count": amount_missing},
]

change_df = pd.DataFrame(change_log)
change_df.to_csv(BASE_DIR / "data" / "cleaning_summary.csv", index=False)
change_df

,dataset,change,count
0,flights,rows in bronze,1005
1,flights,rows in silver,1004
2,flights,airline filled from code,69
3,flights,overnight flights,122
4,passengers,rows in bronze,1039
5,passengers,rows in silver,1000
6,bookings,status set to UNKNOWN,75
7,payments,amount imputed to median,78


## Cell 26 - PreviewA look at the cleaned flights table and the hashed passengers table.

In [25]:
display(clean_data["flights"].drop(columns=["source_row"]).head(10))
display(clean_data["passengers"].drop(columns=["source_row"]).head(10))

,flight_id,airline,source,destination,departure_time,arrival_time,airline_code,is_overnight,duration_minutes
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,SJ,True,174
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,AI,True,108
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,UK,True,105
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,AI,True,156
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,AI,True,299
5,SJ158,SpiceJet,DEL,HYD,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,SJ,True,139
6,6F196,IndiGo,CCU,MAA,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,6F,True,103
7,AI080,Air India,BOM,HYD,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,AI,True,92
8,6F025,IndiGo,BLR,BOM,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,6F,False,55
9,6F251,IndiGo,DEL,BLR,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,6F,False,41


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id
0,P1000,c1016bac4d62c349,0898a357873bdcfd,52,F,90a594d3d2618faa,66d2db762284b8f3,dbc976463c58cebc
1,P1001,64c7e7a78fe65a03,5c012c348379ffe8,15,M,3289671730c1cbb6,eb29bca6561ac5c3,770d0e3801d16861
2,P1002,163bb7b35d57ad33,a1c739a678cb4d19,72,M,1b71dfbe84e5e118,dd17a313cdbb20e0,d94d5c74b1b87dfe
3,P1003,163bb7b35d57ad33,25e29e98d33c67f8,61,F,da1347ef9a577681,a7bb9660ea46cc46,c0365e092e3fd791
4,P1004,6d1eca2f30004fda,4cb38c56eea0ddf5,21,M,6e9695bce76a8f63,26831a7237109534,e15e595420702d9c
5,P1005,d60c08bb8f7409e1,f91d1672b3127a10,83,M,51f8574cd5caced9,fc1a1a94ddee1d5e,f18023e566825a6c
6,P1006,2196452205b86669,84175d4c59410664,87,M,7fcac9b3f1ee8aa9,82c9cec5c9827943,4c6df17148b70b73
7,P1007,8fa9831b6b356cfa,f91d1672b3127a10,75,M,b826f557402f12b1,2024519f3a5797ac,c762e9bd17ddd51c
8,P1008,2196452205b86669,cb9721755c5ee544,75,M,72cc352bcf2c816d,c9716cae22cb547c,83906cf41776b056
9,P1009,dd5395221b5ebee2,ea0e8c0a25bcbe09,88,M,ec230de779bd915b,147667266eb3aa7a,ba24b245313a65c9


## Cell 27 - Handoff to Task 3The silver tables are clean and join safely. Task 3 builds the star schema and thebusiness measures: average flight duration, route traffic, delays and anomalies,and the flight share by airline.

# Task 3 - Data Modelling, Storage and Business KPIsTurn the silver tables into a small star schema in the gold layer, then compute thebusiness measures the brief asks for plus a set of extra operational and revenue KPIs.Everything is written as CSV so Power BI can load it directly.

## Cell 28 - SetupCreate the gold folder and load the four cleaned silver tables.

In [26]:
GOLD_DIR = BASE_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

silver = {name: pd.read_parquet(SILVER_DIR / (name + ".parquet")) for name in RULES}
{name: silver[name].shape for name in silver}

{'flights': (1004, 9),
 'passengers': (1000, 8),
 'bookings': (1000, 9),
 'payments': (1000, 4)}

## Cell 29 - DimensionsBuild the airline, airport, route, date and passenger dimension tables from the silver data.

In [27]:
flights_s = silver["flights"].copy()
flights_s["route"] = flights_s["source"] + "-" + flights_s["destination"]
flights_s["dep_date"] = flights_s["departure_time"].dt.normalize()

dim_airline = (flights_s[["airline_code", "airline"]]
               .drop_duplicates()
               .sort_values("airline_code")
               .reset_index(drop=True))

airports = pd.unique(pd.concat([flights_s["source"], flights_s["destination"]]))
dim_airport = pd.DataFrame({"airport_code": sorted(airports)})

dim_route = (flights_s[["route", "source", "destination"]]
             .drop_duplicates()
             .sort_values("route")
             .reset_index(drop=True))
dim_route.insert(0, "route_id", range(1, len(dim_route) + 1))

booking_dates = pd.to_datetime(silver["bookings"]["booking_date"], format="mixed", errors="coerce").dt.normalize()
all_dates = pd.to_datetime(pd.unique(pd.concat([flights_s["dep_date"], booking_dates]).dropna()))
dim_date = pd.DataFrame({"date": sorted(all_dates)})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["day"] = dim_date["date"].dt.day
dim_date["weekday"] = dim_date["date"].dt.strftime("%a")
dim_date = dim_date[["date_key", "date", "year", "month", "month_name", "day", "weekday"]]

passengers_s = silver["passengers"].copy()
bands = [0, 12, 18, 30, 45, 60, 75, 200]
labels = ["0-12", "13-18", "19-30", "31-45", "46-60", "61-75", "76+"]
passengers_s["age"] = pd.to_numeric(passengers_s["age"], errors="coerce")
passengers_s["age_band"] = pd.cut(passengers_s["age"], bins=bands, labels=labels, right=True)
dim_passenger = passengers_s[["passenger_id", "age", "age_band", "gender"]].reset_index(drop=True)

{"airline": len(dim_airline), "airport": len(dim_airport), "route": len(dim_route), "date": len(dim_date), "passenger": len(dim_passenger)}

{'airline': 4, 'airport': 6, 'route': 30, 'date': 345, 'passenger': 1000}

## Cell 30 - Fact flightsOne row per flight with the route id, date key, departure hour, and the delay and anomalyfields. There is no scheduled time in the source, so delay is measured against the medianduration of the same route.

In [28]:
route_key = dim_route.set_index("route")["route_id"]
flights_s["route_id"] = flights_s["route"].map(route_key)
flights_s["date_key"] = flights_s["dep_date"].dt.strftime("%Y%m%d").astype("Int64")
flights_s["dep_hour"] = flights_s["departure_time"].dt.hour

route_median = flights_s.groupby("route")["duration_minutes"].transform("median")
flights_s["expected_duration"] = route_median.round().astype("Int64")
flights_s["delay_minutes"] = (flights_s["duration_minutes"] - flights_s["expected_duration"]).astype("Int64")
flights_s["is_delayed"] = flights_s["delay_minutes"] > 30
flights_s["is_early"] = flights_s["delay_minutes"] < -30
flights_s["is_severe_delay"] = flights_s["delay_minutes"] > 90

low = flights_s["duration_minutes"].quantile(0.01)
high = flights_s["duration_minutes"].quantile(0.99)
flights_s["is_duration_outlier"] = (flights_s["duration_minutes"] < low) | (flights_s["duration_minutes"] > high)

def anomaly_reason(row):
    reasons = []
    if row["is_severe_delay"]:
        reasons.append("severe_delay")
    if row["is_overnight"]:
        reasons.append("overnight")
    if row["is_duration_outlier"]:
        reasons.append("duration_outlier")
    return ";".join(reasons)

flights_s["anomaly_reason"] = flights_s.apply(anomaly_reason, axis=1)
flights_s["is_anomaly"] = flights_s["anomaly_reason"] != ""

fact_flights = flights_s[["flight_id", "airline_code", "route_id", "date_key", "departure_time",
                          "arrival_time", "dep_hour", "duration_minutes", "expected_duration",
                          "delay_minutes", "is_delayed", "is_early", "is_severe_delay", "is_overnight",
                          "is_duration_outlier", "is_anomaly", "anomaly_reason"]].reset_index(drop=True)
fact_flights.head()

,flight_id,airline_code,route_id,date_key,departure_time,arrival_time,dep_hour,duration_minutes,expected_duration,delay_minutes,is_delayed,is_early,is_severe_delay,is_overnight,is_duration_outlier,is_anomaly,anomaly_reason
0,SJ010,SJ,15,20260420,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,23,174,165,9,False,False,False,True,False,True,overnight
1,AI155,AI,7,20260420,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,23,108,170,-62,False,True,False,True,False,True,overnight
2,UK094,UK,7,20260420,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,23,105,170,-65,False,True,False,True,False,True,overnight
3,AI245,AI,7,20260420,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,23,156,170,-14,False,False,False,True,False,True,overnight
4,AI192,AI,27,20260420,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,23,299,173,126,True,False,True,True,True,True,severe_delay;overnight;duration_outlier


## Cell 31 - Fact bookings and paymentsBookings link a passenger to a flight, payments link an amount to a booking.

In [29]:
bookings_s = silver["bookings"].copy()
bookings_s["booking_date"] = pd.to_datetime(bookings_s["booking_date"], format="mixed", errors="coerce")
bookings_s["date_key"] = bookings_s["booking_date"].dt.strftime("%Y%m%d").astype("Int64")

fact_bookings = bookings_s[["booking_id", "passenger_id", "flight_id", "date_key", "status", "seat_number"]].reset_index(drop=True)

payments_s = silver["payments"].copy()
payments_s["amount"] = pd.to_numeric(payments_s["amount"], errors="coerce")
fact_payments = payments_s[["payment_id", "booking_id", "amount", "payment_method"]].reset_index(drop=True)

{"fact_flights": len(fact_flights), "fact_bookings": len(fact_bookings), "fact_payments": len(fact_payments)}

{'fact_flights': 1004, 'fact_bookings': 1000, 'fact_payments': 1000}

## Cell 32 - Write the star schemaSave the three dimension and three fact tables to the gold folder as CSV.

In [30]:
star = {
    "dim_airline": dim_airline,
    "dim_airport": dim_airport,
    "dim_route": dim_route,
    "dim_date": dim_date,
    "dim_passenger": dim_passenger,
    "fact_flights": fact_flights,
    "fact_bookings": fact_bookings,
    "fact_payments": fact_payments,
}

for name, table in star.items():
    table.to_csv(GOLD_DIR / (name + ".csv"), index=False)
    logger.info("gold %-16s %5d rows  %d cols", name, len(table), table.shape[1])

list(star)

2026-09-06 15:00:43,237  INFO  gold dim_airline          4 rows  2 cols


2026-09-06 15:00:43,241  INFO  gold dim_airport          6 rows  1 cols


2026-09-06 15:00:43,246  INFO  gold dim_route           30 rows  4 cols


2026-09-06 15:00:43,250  INFO  gold dim_date           345 rows  7 cols


2026-09-06 15:00:43,255  INFO  gold dim_passenger     1000 rows  4 cols


2026-09-06 15:00:43,265  INFO  gold fact_flights      1004 rows  17 cols


2026-09-06 15:00:43,272  INFO  gold fact_bookings     1000 rows  6 cols


2026-09-06 15:00:43,279  INFO  gold fact_payments     1000 rows  4 cols


['dim_airline',
 'dim_airport',
 'dim_route',
 'dim_date',
 'dim_passenger',
 'fact_flights',
 'fact_bookings',
 'fact_payments']

## Cell 33 - KPI: average flight durationAverage and median duration overall, by airline and by route.

In [31]:
analysis = fact_flights.merge(dim_airline, on="airline_code", how="left").merge(dim_route, on="route_id", how="left")

kpi_duration_overall = pd.DataFrame([{
    "flights": len(analysis),
    "avg_duration_min": round(analysis["duration_minutes"].mean(), 1),
    "median_duration_min": float(analysis["duration_minutes"].median()),
    "min_duration_min": int(analysis["duration_minutes"].min()),
    "max_duration_min": int(analysis["duration_minutes"].max()),
}])

kpi_duration_by_airline = (analysis.groupby("airline")
                           .agg(flights=("flight_id", "count"),
                                avg_duration_min=("duration_minutes", "mean"),
                                median_duration_min=("duration_minutes", "median"))
                           .round(1).reset_index().sort_values("avg_duration_min", ascending=False))

kpi_duration_by_route = (analysis.groupby(["route", "source", "destination"])
                         .agg(flights=("flight_id", "count"),
                              avg_duration_min=("duration_minutes", "mean"))
                         .round(1).reset_index().sort_values("flights", ascending=False))

kpi_duration_by_airline

,airline,flights,avg_duration_min,median_duration_min
0,Air India,255,165.4,164.0
1,IndiGo,272,165.4,169.0
3,Vistara,230,164.3,166.0
2,SpiceJet,247,163.8,166.0


## Cell 34 - KPI: route-wise trafficFlight count per route with its share of all flights and its delayed share.

In [32]:
kpi_route_traffic = (analysis.groupby(["route", "source", "destination"])
                     .agg(flights=("flight_id", "count"),
                          avg_duration_min=("duration_minutes", "mean"),
                          delayed=("is_delayed", "sum"),
                          overnight=("is_overnight", "sum"))
                     .round(1).reset_index())
kpi_route_traffic["share_pct"] = (100 * kpi_route_traffic["flights"] / kpi_route_traffic["flights"].sum()).round(1)
kpi_route_traffic["delayed_pct"] = (100 * kpi_route_traffic["delayed"] / kpi_route_traffic["flights"]).round(1)
kpi_route_traffic = kpi_route_traffic.sort_values("flights", ascending=False).reset_index(drop=True)
kpi_route_traffic.head(10)

,route,source,destination,flights,avg_duration_min,delayed,overnight,share_pct,delayed_pct
0,BOM-CCU,BOM,CCU,90,169.5,32,16,9.0,35.6
1,CCU-DEL,CCU,DEL,72,153.6,30,11,7.2,41.7
2,MAA-BLR,MAA,BLR,65,172.8,28,8,6.5,43.1
3,BLR-BOM,BLR,BOM,60,147.7,22,7,6.0,36.7
4,HYD-MAA,HYD,MAA,57,152.8,21,5,5.7,36.8
5,DEL-HYD,DEL,HYD,54,174.8,20,6,5.4,37.0
6,HYD-DEL,HYD,DEL,42,185.4,15,9,4.2,35.7
7,BOM-DEL,BOM,DEL,39,153.8,12,3,3.9,30.8
8,CCU-BOM,CCU,BOM,33,164.0,13,1,3.3,39.4
9,DEL-BLR,DEL,BLR,29,170.0,10,3,2.9,34.5


## Cell 35 - KPI: delays and anomaliesCounts of delayed, early, overnight and outlier flights overall and by airline.

In [33]:
kpi_anomaly_overall = pd.DataFrame([{
    "flights": len(analysis),
    "delayed": int(analysis["is_delayed"].sum()),
    "early": int(analysis["is_early"].sum()),
    "overnight": int(analysis["is_overnight"].sum()),
    "duration_outlier": int(analysis["is_duration_outlier"].sum()),
    "any_anomaly": int(analysis["is_anomaly"].sum()),
    "anomaly_pct": round(100 * analysis["is_anomaly"].mean(), 1),
}])

kpi_anomaly_by_airline = (analysis.groupby("airline")
                          .agg(flights=("flight_id", "count"),
                               delayed=("is_delayed", "sum"),
                               overnight=("is_overnight", "sum"),
                               anomalies=("is_anomaly", "sum"))
                          .reset_index())
kpi_anomaly_by_airline["delayed_pct"] = (100 * kpi_anomaly_by_airline["delayed"] / kpi_anomaly_by_airline["flights"]).round(1)
kpi_anomaly_by_airline["anomaly_pct"] = (100 * kpi_anomaly_by_airline["anomalies"] / kpi_anomaly_by_airline["flights"]).round(1)

kpi_anomaly_reasons = (fact_flights[fact_flights["anomaly_reason"] != ""]
                       .assign(reason=lambda d: d["anomaly_reason"].str.split(";"))
                       .explode("reason")
                       .groupby("reason").size().reset_index(name="flights")
                       .sort_values("flights", ascending=False))

kpi_anomaly_overall

,flights,delayed,early,overnight,duration_outlier,any_anomaly,anomaly_pct
0,1004,374,372,122,22,269,26.8


## Cell 36 - KPI: distribution of flights by airlineFlight count and percentage share per airline with average duration and delayed share.

In [34]:
kpi_flights_by_airline = (analysis.groupby("airline")
                          .agg(flights=("flight_id", "count"),
                               avg_duration_min=("duration_minutes", "mean"),
                               overnight=("is_overnight", "sum"),
                               delayed=("is_delayed", "sum"))
                          .round(1).reset_index())
kpi_flights_by_airline["share_pct"] = (100 * kpi_flights_by_airline["flights"] / kpi_flights_by_airline["flights"].sum()).round(1)
kpi_flights_by_airline = kpi_flights_by_airline.sort_values("flights", ascending=False).reset_index(drop=True)
kpi_flights_by_airline

,airline,flights,avg_duration_min,overnight,delayed,share_pct
0,IndiGo,272,165.4,32,113,27.1
1,Air India,255,165.4,33,93,25.4
2,SpiceJet,247,163.8,29,87,24.6
3,Vistara,230,164.3,28,81,22.9


## Cell 37 - KPI: peak hours and airport trafficFlights by departure hour, and departures plus arrivals per airport.

In [35]:
kpi_peak_hours = (fact_flights.groupby("dep_hour").size().reset_index(name="flights").sort_values("dep_hour"))

departures = fact_flights.merge(dim_route, on="route_id", how="left").groupby("source").size()
arrivals = fact_flights.merge(dim_route, on="route_id", how="left").groupby("destination").size()
kpi_airport_traffic = pd.DataFrame({"departures": departures, "arrivals": arrivals}).fillna(0).astype(int)
kpi_airport_traffic["total"] = kpi_airport_traffic["departures"] + kpi_airport_traffic["arrivals"]
kpi_airport_traffic = kpi_airport_traffic.reset_index(names="airport").sort_values("total", ascending=False).reset_index(drop=True)
kpi_airport_traffic

,airport,departures,arrivals,total
0,BOM,205,169,374
1,DEL,160,198,358
2,CCU,169,187,356
3,HYD,178,141,319
4,MAA,157,147,304
5,BLR,135,162,297


## Cell 38 - KPI: revenueJoin payments to bookings to flights, then total revenue and average fare by airline,by route and by payment method. Cancelled bookings are shown separately.

In [36]:
revenue = (fact_payments
           .merge(fact_bookings[["booking_id", "flight_id", "status"]], on="booking_id", how="left")
           .merge(fact_flights[["flight_id", "airline_code", "route_id"]], on="flight_id", how="left")
           .merge(dim_airline, on="airline_code", how="left")
           .merge(dim_route[["route_id", "route"]], on="route_id", how="left"))

kpi_revenue_by_airline = (revenue.groupby("airline")
                          .agg(revenue=("amount", "sum"), payments=("payment_id", "count"), avg_fare=("amount", "mean"))
                          .round(0).reset_index().sort_values("revenue", ascending=False))

kpi_revenue_by_route = (revenue.groupby("route")
                        .agg(revenue=("amount", "sum"), payments=("payment_id", "count"), avg_fare=("amount", "mean"))
                        .round(0).reset_index().sort_values("revenue", ascending=False))

kpi_revenue_by_method = (revenue.groupby("payment_method")
                         .agg(revenue=("amount", "sum"), payments=("payment_id", "count"), avg_fare=("amount", "mean"))
                         .round(0).reset_index().sort_values("revenue", ascending=False))

kpi_revenue_by_status = (revenue.groupby("status")
                         .agg(revenue=("amount", "sum"), payments=("payment_id", "count"))
                         .round(0).reset_index())

kpi_revenue_by_airline

,airline,revenue,payments,avg_fare
3,Vistara,2299374.0,273,8423.0
2,SpiceJet,2026768.0,254,7979.0
0,Air India,1903937.0,247,7708.0
1,IndiGo,1781179.0,226,7881.0


## Cell 39 - KPI: booking statusBooking count and share per status, and the cancellation rate.

In [37]:
kpi_booking_status = (fact_bookings.groupby("status").size().reset_index(name="bookings"))
kpi_booking_status["share_pct"] = (100 * kpi_booking_status["bookings"] / kpi_booking_status["bookings"].sum()).round(1)

cancellation_rate = round(100 * (fact_bookings["status"] == "CANCELLED").mean(), 1)
kpi_booking_status, cancellation_rate

(      status  bookings  share_pct
 0  CANCELLED       314       31.4
 1  CONFIRMED       320       32.0
 2    PENDING       291       29.1
 3    UNKNOWN        75        7.5,
 np.float64(31.4))

## Cell 40 - OverviewA single headline row for the dashboard KPI cards.

In [38]:
kpi_overview = pd.DataFrame([{
    "total_flights": len(fact_flights),
    "airlines": dim_airline["airline"].nunique(),
    "routes": len(dim_route),
    "avg_duration_min": round(fact_flights["duration_minutes"].mean(), 1),
    "overnight_flights": int(fact_flights["is_overnight"].sum()),
    "delayed_flights": int(fact_flights["is_delayed"].sum()),
    "anomaly_flights": int(fact_flights["is_anomaly"].sum()),
    "total_bookings": len(fact_bookings),
    "cancellation_rate_pct": cancellation_rate,
    "total_revenue": round(fact_payments["amount"].sum(), 0),
    "avg_fare": round(fact_payments["amount"].mean(), 0),
}])
kpi_overview

,total_flights,airlines,routes,avg_duration_min,overnight_flights,delayed_flights,anomaly_flights,total_bookings,cancellation_rate_pct,total_revenue,avg_fare
0,1004,4,30,164.8,122,374,269,1000,31.4,8011258.0,8011.0


## Cell 41 - Write the KPI tablesSave every KPI table to the gold folder as CSV.

In [39]:
kpi_tables = {
    "kpi_overview": kpi_overview,
    "kpi_duration_overall": kpi_duration_overall,
    "kpi_duration_by_airline": kpi_duration_by_airline,
    "kpi_duration_by_route": kpi_duration_by_route,
    "kpi_route_traffic": kpi_route_traffic,
    "kpi_anomaly_overall": kpi_anomaly_overall,
    "kpi_anomaly_by_airline": kpi_anomaly_by_airline,
    "kpi_anomaly_reasons": kpi_anomaly_reasons,
    "kpi_flights_by_airline": kpi_flights_by_airline,
    "kpi_peak_hours": kpi_peak_hours,
    "kpi_airport_traffic": kpi_airport_traffic,
    "kpi_revenue_by_airline": kpi_revenue_by_airline,
    "kpi_revenue_by_route": kpi_revenue_by_route,
    "kpi_revenue_by_method": kpi_revenue_by_method,
    "kpi_revenue_by_status": kpi_revenue_by_status,
    "kpi_booking_status": kpi_booking_status,
}

for name, table in kpi_tables.items():
    table.to_csv(GOLD_DIR / (name + ".csv"), index=False)

logger.info("wrote %d kpi tables to %s", len(kpi_tables), GOLD_DIR)
list(kpi_tables)

2026-09-06 15:00:43,656  INFO  wrote 16 kpi tables to C:\Users\angel\Documents\Neostats\data\gold


['kpi_overview',
 'kpi_duration_overall',
 'kpi_duration_by_airline',
 'kpi_duration_by_route',
 'kpi_route_traffic',
 'kpi_anomaly_overall',
 'kpi_anomaly_by_airline',
 'kpi_anomaly_reasons',
 'kpi_flights_by_airline',
 'kpi_peak_hours',
 'kpi_airport_traffic',
 'kpi_revenue_by_airline',
 'kpi_revenue_by_route',
 'kpi_revenue_by_method',
 'kpi_revenue_by_status',
 'kpi_booking_status']

## Cell 42 - PreviewThe overview row and the main KPI tables for a quick check.

In [40]:
display(kpi_overview)
display(kpi_flights_by_airline)
display(kpi_route_traffic.head(10))
display(kpi_anomaly_by_airline)
display(kpi_revenue_by_airline)

,total_flights,airlines,routes,avg_duration_min,overnight_flights,delayed_flights,anomaly_flights,total_bookings,cancellation_rate_pct,total_revenue,avg_fare
0,1004,4,30,164.8,122,374,269,1000,31.4,8011258.0,8011.0


,airline,flights,avg_duration_min,overnight,delayed,share_pct
0,IndiGo,272,165.4,32,113,27.1
1,Air India,255,165.4,33,93,25.4
2,SpiceJet,247,163.8,29,87,24.6
3,Vistara,230,164.3,28,81,22.9


,route,source,destination,flights,avg_duration_min,delayed,overnight,share_pct,delayed_pct
0,BOM-CCU,BOM,CCU,90,169.5,32,16,9.0,35.6
1,CCU-DEL,CCU,DEL,72,153.6,30,11,7.2,41.7
2,MAA-BLR,MAA,BLR,65,172.8,28,8,6.5,43.1
3,BLR-BOM,BLR,BOM,60,147.7,22,7,6.0,36.7
4,HYD-MAA,HYD,MAA,57,152.8,21,5,5.7,36.8
5,DEL-HYD,DEL,HYD,54,174.8,20,6,5.4,37.0
6,HYD-DEL,HYD,DEL,42,185.4,15,9,4.2,35.7
7,BOM-DEL,BOM,DEL,39,153.8,12,3,3.9,30.8
8,CCU-BOM,CCU,BOM,33,164.0,13,1,3.3,39.4
9,DEL-BLR,DEL,BLR,29,170.0,10,3,2.9,34.5


,airline,flights,delayed,overnight,anomalies,delayed_pct,anomaly_pct
0,Air India,255,93,33,62,36.5,24.3
1,IndiGo,272,113,32,76,41.5,27.9
2,SpiceJet,247,87,29,67,35.2,27.1
3,Vistara,230,81,28,64,35.2,27.8


,airline,revenue,payments,avg_fare
3,Vistara,2299374.0,273,8423.0
2,SpiceJet,2026768.0,254,7979.0
0,Air India,1903937.0,247,7708.0
1,IndiGo,1781179.0,226,7881.0


## Cell 43 - Outputs for reportingThe gold folder now holds the star schema and the KPI tables as CSV. These feed thePower BI report, which is built separately in Power BI Desktop followingdocs/powerbi_build_steps.md.